# Metajeux-01 : Open Games et Lentilles -- la representation locale qui modifie le global dont elle est issue

**Navigation** : [<< 17-MultiAgent-RL](../GameTheory-17-MultiAgent-RL-CSharp.ipynb) | [Version Python](Metajeux-01-Open-Games-Lentilles-Python.ipynb) | [Index](README.md)

**Kernel** : .NET C#

***

## Concept

Le depot **emploie deja les mots** -- << lentille >>, << composer des regards >> -- sans les avoir compris.
C'est une dette de vocabulaire, et elle se solde avant de continuer a s'en servir.

Une **lentille** est une paire :
- un `view` qui **extrait** une partie d'un tout
- un `update` qui **reinjecte** une modification de la partie dans le tout

Ce n'est PAS une projection. C'est ce qui autorise la phrase qui interesse ICT :

> une representation locale n'est pas seulement **extraite** du global ; elle peut ensuite **MODIFIER** le global dont elle etait issue.

Les **open games** de Hedges ajoutent la teleologie : un jeu ouvert a une frontiere (entrees/sorties et leurs contreparties) et une **fonction de meilleure reponse** ; deux jeux ouverts se composent en serie et en parallele, et la composition **preserve** la structure d'equilibre.

**References** :
- Hedges, J. (2017). *Coherence for lenses and open games.* arXiv:1704.02230.
- Ghani, N., Hedges, J., Winschel, V., Zahn, P. (2018). *Compositional game theory.* LICS 2018, arXiv:1603.04641.

**Limites** :
- Les resultats de Hedges sont **RAPPORTES**, pas re-prouves. Le notebook mesure sur ses propres cas.
- La composition est ici au niveau jouet : deux jeux ouverts a deux strategies, dont l'avant composee est
  verifiee sur trois entrees et la BR chainee evaluee au seul cas temoin `x = 0` -- pas une preuve de la
  theorie des open games.


In [1]:
using System;
using System.Collections.Generic;

// World W = (a, b), avec affichage compatible avec les notations du cours.
public sealed record World(int A, int B)
{
    public override string ToString() => $"({A}, {B})";
}

// Port manuel de numpy.random.default_rng : generateur .NET natif a graine fixe.
var RNG = new Random(20260822);

// Port manuel de typing : les types generiques BCL remplacent les alias Python.
Func<int, int> identity = value => value;

Console.WriteLine($"BCL .NET {Environment.Version} ; RNG seed=20260822");

BCL .NET 10.0.13 ; RNG seed=20260822


## Section 1 -- Lentille : view, update, lois get-put/put-get

**Definition (Hedges,)** : une lentille `L : W <= P` entre un tout `W` et une partie `P` est une paire
`(view, update)` :
- `view : W -> P` extrait la partie
- `update : (W, P) -> W` reinjecte une partie modifiee

**Lois verifiees sur tout `w : W` et `p : P`** :
- `get-put` : `update(w, view(w)) == w` (l'extraction suivie d'une reinjection identique laisse le tout intact)
- `put-get` : `view(update(w, p)) == p` (apres reinjection, l'extraction rend exactement la partie reinjectee)

**Une lentille qui viole une loi n'est PAS une lentille**, c'est une fonction. L'exemple guide ci-dessous exhibe un contre-exemple.


In [2]:
// Implementation de reference -- une lentille CONFORME sur un world W = (a, b) et une partie P = a

public sealed class Lens<TWorld, TPart>
{
    public Func<TWorld, TPart> View { get; }
    public Func<TWorld, TPart, TWorld> Update { get; }
    public string Name { get; }

    public Lens(Func<TWorld, TPart> view, Func<TWorld, TPart, TWorld> update, string name = "lens")
    {
        View = view;
        Update = update;
        Name = name;
    }

    public TPart Get(TWorld world) => View(world);
    public TWorld Put(TWorld world, TPart part) => Update(world, part);
}

static bool get_put_holds<TWorld, TPart>(Lens<TWorld, TPart> lens, TWorld world)
{
    var extracted = lens.View(world);
    return EqualityComparer<TWorld>.Default.Equals(lens.Update(world, extracted), world);
}

static bool get_put_holds<TWorld, TPart>(Lens<TWorld, TPart> lens, TWorld world, TPart part)
{
    return EqualityComparer<TWorld>.Default.Equals(lens.Update(world, part), world);
}

static bool put_get_holds<TWorld, TPart>(Lens<TWorld, TPart> lens, TWorld world, TPart part)
{
    return EqualityComparer<TPart>.Default.Equals(lens.View(lens.Update(world, part)), part);
}

// World W = (a:int, b:int), partie P = a (le premier element du tuple)
int view_first(World world) => world.A;
World update_first(World world, int part) => new(part, world.B);

var L1 = new Lens<World, int>(view_first, update_first, "first-of-2");

// Verification numerique sur 5 worlds et 5 parts
bool ok_gp = true;
bool ok_pg = true;
var worlds = new[]
{
    new World(1, 2),
    new World(5, 0),
    new World(-3, 7),
    new World(10, -1),
    new World(0, 0),
};
var parts = new[] { 10, 0, -5, 99, 1 };

foreach (var world in worlds)
{
    if (!get_put_holds(L1, world))
        ok_gp = false;

    foreach (var part in parts)
    {
        if (!put_get_holds(L1, world, part))
            ok_pg = false;
    }
}

Console.WriteLine($"L1 (premiere composante d un tuple) : get-put={ok_gp}, put-get={ok_pg}");

L1 (premiere composante d un tuple) : get-put=True, put-get=True


### Exemple guidé — Falsifier une loi

La lentille `L1` satisfait les deux lois sur les cas testés. Pour comprendre pourquoi chacune est nécessaire, l'exemple suivant conserve le même type de monde mais introduit volontairement une réinjection qui décale la valeur observée.


In [3]:
// Contre-exemple : une pseudo-lentille qui VIOLE get-put.
// 
// Cas : world W = (a, b), pseudo-partie P = a + 1 (view calcule a+1, update pose a' = p - 1).
// 
// view(w) = w[0] + 1
// update(w, p) = (p - 1, w[1])
// 
// get-put : update(w, view(w)) = update(w, w[0]+1) = (w[0]+1-1, w[1]) = (w[0], w[1]) = w. OK ici.
// put-get : view(update(w, p)) = (p - 1) + 1 = p. OK ici.
// 
// Cette pseudo-lentille respecte les deux lois -- pas un contre-exemple. Il faut aller plus loin.

// Vrai contre-exemple : world W = (a:int, b:int), pseudo-partie P = a (meme type), mais
// view(w) = w[0]   (OK : renvoie bien la partie 'a')
// update(w, p) = (p + 1, w[1])  <-- AJOUTE 1 a la valeur reinjectee
// 
// get-put : update(w, view(w)) = (w[0] + 1, w[1]) != w.  <-- LOI VIOLEE
int view_bad(World world) => world.A;
World update_bad(World world, int part) => new(part + 1, world.B); // ajoute 1 : brise get-put

var L_bad = new Lens<World, int>(view_bad, update_bad, "bad-lens-plus-1");

var w_test = new World(5, 2);
var p_extracted = L_bad.View(w_test);
var w_after = L_bad.Update(w_test, p_extracted);
Console.WriteLine($"w_test = {w_test}, view(w) = {p_extracted}, update(w, view(w)) = {w_after}");
Console.WriteLine($"get-put violé : {!w_after.Equals(w_test)} (devrait etre True si loi violee)");

w_test = (5, 2), view(w) = 5, update(w, view(w)) = (6, 2)


get-put violé : True (devrait etre True si loi violee)


### Lecture de la lentille

**Mesure** :
- `L1` (premiere composante d'un tuple 2) : `get-put=True` sur 5 worlds, `put-get=True` sur 25 tests (5 worlds x 5 parts).
- `L_bad` (avec `update(w, p) = (p+1, w[1])`) : `get-put` viole systematiquement.

**Pedagogie** : les lois d'une lentille ne sont pas decoratives. Une paire `view/update` qui viole l'une d'elles
n'est pas une lentille au sens de Hedges -- c'est une fonction qui **parle de la structure** sans la **respecter**.
Le contre-exemple ci-dessus exhibe une violation sur 1 cas (`w = (5, 2)`), ce qui suffit : la loi est universelle,
sa violation sur 1 cas la falsifie.


### Exercice 1 — Isoler la loi put-get

**Contexte.** L'exemple guidé `L_bad` falsifie `get-put`. Les deux lois sont indépendantes : une paire `view/update` peut préserver le monde quand elle réinjecte la partie observée, tout en ignorant une nouvelle partie proposée.

**Objectif.** Construisez une pseudo-lentille `L_pg` qui satisfait `get-put` mais viole `put-get`. Vérifiez les deux résultats avec `get_put_holds` et `put_get_holds` sur `w = (5, 2)` et `p = 9`.

**Indices.**

1. Gardez `view_pg(w) = w[0]`.
2. Cherchez un `update_pg` qui laisse toujours `w` inchangé, quelle que soit la valeur de `p`.
3. Affichez séparément le verdict de chaque loi pour rendre le contre-exemple vérifiable.


In [4]:
// Exercice 1 a completer
// Etape 1 : definir view_pg et update_pg pour isoler la loi put-get.
// Etape 2 : construire L_pg avec Lens.
// Etape 3 : verifier get-put et put-get sur w=(5, 2), p=9.
// Indice : un update qui ignore p peut conserver le monde sans reinjecter la nouvelle partie.

// TODO etudiant : view_pg doit renvoyer la premiere composante du monde, et update_pg
// doit laisser le monde intact quelle que soit la valeur de p.
static Lens<World, int>? ConstruireLpg()
{
    return null;
}

var L_pg = ConstruireLpg();
var resultat_ex1 = L_pg;

if (L_pg is null)
{
    Console.WriteLine("Exercice 1 a completer");
}
else
{
    var lens = L_pg;
    var w_ex1 = new World(5, 2);
    int p_ex1 = 9;
    Console.WriteLine($"get-put : {get_put_holds(lens, w_ex1)}");
    Console.WriteLine($"put-get : {put_get_holds(lens, w_ex1, p_ex1)}");
}


Exercice 1 a completer



(9,24): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## Section 2 -- Open games : composition en serie et BR composee

**Open game (Hedges)** : un objet `(S, R, X, Y, F, br)` :
- `S, R` : espaces de strategies et de realisations (decision/observation)
- `X, Y` : espaces de frontiere (inputs/outputs)
- `F : (X, R, S) -> Y` : fonction d'avant (forward)
- `br : (Y, R, X) -> S` : meilleure reponse (backward)

**Composition serie** : `G2 o G1` -- la sortie de `G1` alimente l'entree de `G2`. **La BR de la composition
se deduit des BR des composants** : `br_compose(x, r) = br_2(br_1(...), r_2, x)` (backward chaining).

**Resultat cle de Hedges** : la composition preserve les equilibres de Nash. Ce resultat est **RAPPORTE**
(cf. limites) : le carnet ne calcule aucun equilibre de Nash. Ce qui est verifie ici, c'est la chaine des
BR sur les jeux ouverts triviaux `G1` et `G2` -- au cas temoin `x = 0`.


In [5]:
// Implementation d'un open game jouet : un joueur choisissant entre 2 actions en fonction d'une entree.

using System.Globalization;

// Affichage invariant (la culture FR ne persiste pas entre cellules .NET Interactive)
static string S(double v) => v.ToString("0.0", CultureInfo.InvariantCulture);
static string Sgn(double v) => (v >= 0 ? "+" : "-") + Math.Abs(v).ToString("0.0", CultureInfo.InvariantCulture);

public sealed class OpenGame
{
    public string Name { get; }
    public Func<double, double> Forward { get; }              // F : X -> Y
    public Func<double, int> BestResponse { get; }              // br : Y -> S
    public Func<double, (int, int)>? ChainedBestResponse { get; } // br chainee (backward chaining)

    public OpenGame(string name, Func<double, double> forward, Func<double, int> bestResponse,
                    Func<double, (int, int)>? chained = null)
    {
        Name = name;
        Forward = forward;
        BestResponse = bestResponse;
        ChainedBestResponse = chained;
    }

    public override string ToString() => $"<OpenGame {Name}>";
}

// Open game 1 : le joueur choisit entre cooperation (s=0) ou defection (s=1) selon une entree x.
// F_1(x) = x + 1 (l'avant amplifie l'entree)
// br_1(y) = 0 si y > 0.5 sinon 1 (cooperation si signal positif)
double F1(double x) => x + 1;
int br1(double y) => y > 0.5 ? 0 : 1;

var G1 = new OpenGame("G1-cooperate-if-positive", F1, br1);

// Open game 2 : le joueur ajuste une intensite selon la sortie de G1.
// F_2(y) = y * 2 (amplifie la sortie de G1)
// br_2(z) = 1 si z > 1.5 sinon 0 (engagement si gros signal)
double F2(double y) => y * 2;
int br2(double z) => z > 1.5 ? 1 : 0;

var G2 = new OpenGame("G2-commit-if-large", F2, br2);

// Composition serie : G2 o G1 -- la sortie de G1 alimente G2.
static OpenGame compose_series(OpenGame g2, OpenGame g1)
{
    // Composition serie : g2.forward prend la sortie de g1.forward comme entree.
    double F_compose(double x) => g2.Forward(g1.Forward(x));
    // backward chaining (Hedges) : la BR de g2 sert d'abord, puis celle de g1.
    // Pour la demonstration : on evalue juste les BR sur leur domaine respectif.
    (int, int) br_chain(double x) => (g1.BestResponse(g1.Forward(x)), g2.BestResponse(g2.Forward(g1.Forward(x))));
    return new OpenGame($"{g2.Name} o {g1.Name}", F_compose, y => g2.BestResponse(y), br_chain);
}

var G_compose = compose_series(G2, G1);

// Test sur 3 entrees
foreach (double x in new[] { -1.0, 0.0, 0.5 })
{
    double y1 = G1.Forward(x);
    double z = G_compose.Forward(x);
    Console.WriteLine($"x={S(x)} -> G1.forward={Sgn(y1)} -> G_compose.forward={Sgn(z)}");
}


x=-1.0 -> G1.forward=+0.0 -> G_compose.forward=+0.0


x=0.0 -> G1.forward=+1.0 -> G_compose.forward=+2.0


x=0.5 -> G1.forward=+1.5 -> G_compose.forward=+3.0



(17,45): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(14,36): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



**Lecture chiffree — trois points pour deux frontieres.** La sortie aligne trois entrees : `x=-1.0 -> G1.forward=+0.0 -> G_compose.forward=+0.0`, `x=+0.0 -> G1.forward=+1.0 -> G_compose.forward=+2.0`, `x=+0.5 -> G1.forward=+1.5 -> G_compose.forward=+3.0`. Deux regles se lisent colonne par colonne : G1 fait `x + 1` (0.0, 1.0, 1.5 sur les trois lignes), G2 fait `2 y` (0.0, 2.0, 3.0) — et la composee `2 (x + 1)` verifie ses trois lignes d'un coup : 2 x (-1+1) = 0, 2 x (0+1) = 2, 2 x (0.5+1) = 3. La lecture complete de la table va au-dela du cas temoin `x = 0` que la section suivante reprend : les trois points tracent la frontiere avant entiere de la composition, et sa linearite est la raison pour laquelle la BR chainee restera definie partout — la propriete de Hedges, dont la verification explicite sur le cas temoin `x = 0` est reprise plus bas dans le carnet.


### Vérification de la meilleure réponse composée

Le calcul précédent suit la frontière avant, de `x` jusqu'à la sortie composée. La cellule suivante reprend un cas témoin et évalue séparément les meilleures réponses locales sur les sorties effectivement obtenues.


In [6]:
// Verification : la BR de la composition = composition des BR (propriete de Hedges).
// 
// Cas x = 0.0 :
//   y1 = G1.forward(0) = 1.0
//   z = G2.forward(1.0) = 2.0
//   br_1(y1=1.0) = 0 (signal positif, cooperation)
//   br_2(z=2.0) = 1 (gros signal, engagement)
// 
// Cas x = -1.0 :
//   y1 = G1.forward(-1) = 0.0
//   z = G2.forward(0.0) = 0.0
//   br_1(0.0) = 1 (signal non positif, defection)
//   br_2(0.0) = 0 (signal faible, pas d engagement)

double x = 0.0;
double y1 = G1.Forward(x);
double z = G_compose.Forward(x);
var br_chain = (G1.BestResponse(y1), G2.BestResponse(z));
Console.WriteLine($"x = {S(x)}, F1(x) = {S(y1)}, F_compose(x) = {S(z)}");
Console.WriteLine($"br chain (s1, s2) = ({br_chain.Item1}, {br_chain.Item2})");
Console.WriteLine("BR de la composition = BR de chaque composant evaluee sur sa sortie locale. C est la propriete de Hedges.");


x = 0.0, F1(x) = 1.0, F_compose(x) = 2.0


br chain (s1, s2) = (0, 1)


BR de la composition = BR de chaque composant evaluee sur sa sortie locale. C est la propriete de Hedges.


### Lecture de la composition

**Mesure** : pour `x = 0`, `F_compose(0) = 2.0`. La BR chainee `(br_1(1.0), br_2(2.0)) = (0, 1)` --
cooperation en amont, engagement en aval. La BR de la composition **n'est pas une nouvelle fonction** :
c'est la **composition point par point** des BR des composants.

**Pedagogie** : c'est cette preservation qui rend les open games composables. Sans elle, composer deux
jeux ouverts serait un nouveau jeu, pas un jeu derivé. Hedges demontre que la composition preserve
les equilibres de Nash ; on **constate** ici sur le cas jouet que les BR restent definies.


### Exercice 2 — Composer deux jeux en parallèle

**Contexte.** La composition en série fait circuler la sortie de `G1` vers `G2`. En parallèle, les deux jeux gardent des frontières indépendantes : aucun résultat local n'alimente l'autre jeu.

**Objectif.** Complétez `compose_parallel(g2, g1)` pour que son `forward` transforme une paire `(x1, x2)` en `(g1.forward(x1), g2.forward(x2))`. Testez ensuite la composition sur au moins deux paires d'entrées et évaluez les deux meilleures réponses locales.

**Indices.**

1. Le `forward` reçoit et renvoie des tuples de deux éléments.
2. Réutilisez la classe `OpenGame` et les objets `G1`, `G2` déjà définis.
3. Contrairement à la série, n'appelez jamais `g2.forward` avec la sortie de `g1.forward`.


In [7]:
// Exercice 2 a completer
// Etape 1 : definir compose_parallel(g2, g1) avec des frontieres independantes.
// Etape 2 : tester le forward sur (-1.0, 0.0) et (0.5, 1.0).
// Etape 3 : evaluer les meilleures reponses locales sur chaque sortie.
// Indice : aucune sortie locale ne devient l'entree de l'autre jeu.

// TODO etudiant : le forward doit transformer (x1, x2) en (g1.forward(x1), g2.forward(x2)),
// et les meilleures reponses locales s'evaluer separement sur chaque sortie.
public sealed class ParallelGame
{
    public string Name { get; }
    public Func<(double, double), (double, double)> Forward { get; }
    public Func<(double, double), (int, int)> BestResponses { get; }

    public ParallelGame(string name,
                        Func<(double, double), (double, double)> forward,
                        Func<(double, double), (int, int)> bestResponses)
    {
        Name = name;
        Forward = forward;
        BestResponses = bestResponses;
    }
}

static ParallelGame? compose_parallel(OpenGame g2, OpenGame g1)
{
    return null;
}

var G_parallel = compose_parallel(G2, G1);
var resultat_ex2 = G_parallel;

if (G_parallel is null)
{
    Console.WriteLine("Exercice 2 a completer");
}
else
{
    foreach (var pair in new[] { (-1.0, 0.0), (0.5, 1.0) })
    {
        var sortie = G_parallel.Forward(pair);
        var br = G_parallel.BestResponses(sortie);
        Console.WriteLine($"({pair.Item1}, {pair.Item2}) -> ({sortie.Item1}, {sortie.Item2}) ; br=({br.Item1}, {br.Item2})");
    }
}


Exercice 2 a completer



(25,20): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## Section 3 -- Boucle : l'update modifie le global, la BR change en consequence

**La phrase qui interesse ICT** : une representation locale n'est pas seulement **extraite** du global ;
elle peut ensuite **MODIFIER** le global dont elle etait issue. C'est la boucle `W -> P -> W'`.

**Exemple guide** : on a un world `W = (a, b)`. Une lentille extrait `P = a`. Un agent prend `P`, calcule
une decision `d(P)`. La lentille reinjecte `a' = d(a)`. **Le world change**. La BR du tour suivant
depend du nouveau `a'`.

On mesure : point fixe (decision = BR coherente) ou absence de point fixe (oscillation / divergence).


In [8]:
// Boucle : world W = (a, b), lentille premiere composante, agent decide a' = clip(a + delta, -1, 1).
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Clip : equivalent numpy.clip, ecrit a la main (port manuel, aucun NuGet).
static double Clip(double v, double lo, double hi) => v < lo ? lo : (v > hi ? hi : v);

// Decision : a' = clip(a + delta, -1, 1).
static double decision_function(double a, double delta) => Clip(a + delta, -1.0, 1.0);

// Itere la boucle W -> P -> decision -> update -> W' -> ...
static List<double> iterate_loop(double a0, double b, double delta, int nSteps)
{
    double a = a0;
    var history = new List<double> { a };
    for (int k = 0; k < nSteps; k++)
    {
        a = decision_function(a, delta);
        history.Add(a);
    }
    return history;
}

// Affichage invariant des trajectoires (une decimale, comme l'affichage Python).
static string Traj(IEnumerable<double> xs) =>
    "[" + string.Join(", ", xs.Select(v => v.ToString("0.0", CultureInfo.InvariantCulture))) + "]";

// Cas 1 : delta = 0.5 (tire vers 1). Partant de a0 = 0.5, on tend vers 1 (point fixe a* = 1).
var h1 = iterate_loop(a0: 0.5, b: 0, delta: 0.5, nSteps: 10);
Console.WriteLine($"Cas 1 (delta=0.5, a0=0.5) : 10 etapes -> {Traj(h1)}");
Console.WriteLine("  -> converge vers 1.0 (point fixe a* = 1.0)");

// Cas 2 : delta = -0.3 (tire vers -1). Partant de a0 = 0.5, on descend vers ... mais le clip empeche de descendre si la valeur devient < -1.
// point fixe : a* tel que a* + delta = a* -> delta = 0. Impossible. Donc le systeme converge vers -1 (limite du clip) ou oscille.
var h2 = iterate_loop(a0: 0.5, b: 0, delta: -0.3, nSteps: 10);
Console.WriteLine($"Cas 2 (delta=-0.3, a0=0.5) : 10 etapes -> {Traj(h2)}");
Console.WriteLine("  -> converge vers -1.0 (point fixe par clip)");

// Cas 3 : delta oscillant. Decision depend du signe de a. Pas de point fixe (le systeme oscille).
static double decision_oscillating(double a) => -a; // oscille : a' = -a -> a'' = a -> divergence sign alternant

double a3 = 0.5;
var h3 = new List<double> { a3 };
for (int k = 0; k < 6; k++)
{
    a3 = decision_oscillating(a3);
    h3.Add(a3);
}
Console.WriteLine($"Cas 3 (decision oscillante a -> -a) : 6 etapes -> {Traj(h3)}");
Console.WriteLine("  -> oscille, point fixe a* = 0 (trivial, instable). Pas de convergence interessante.");


Cas 1 (delta=0.5, a0=0.5) : 10 etapes -> [0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


  -> converge vers 1.0 (point fixe a* = 1.0)


Cas 2 (delta=-0.3, a0=0.5) : 10 etapes -> [0.5, 0.2, -0.1, -0.4, -0.7, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]


  -> converge vers -1.0 (point fixe par clip)


Cas 3 (decision oscillante a -> -a) : 6 etapes -> [0.5, -0.5, 0.5, -0.5, 0.5, -0.5, 0.5]


  -> oscille, point fixe a* = 0 (trivial, instable). Pas de convergence interessante.


### Mesurer l'approche des états limites

Les trajectoires distinguent déjà convergence et oscillation. La mesure suivante transforme cette lecture qualitative en distances successives aux deux états limites imposés par le clipping.


In [9]:
// Mesure quantitative : distance au point fixe theorique sur 20 iterations.
using System.Globalization;
using System.Linq;

// Cas 1 (delta=0.5) : point fixe theorique a* = min(1.0, max(-1.0, 0.5 + 0.5)) = 1.0.
var h1_20 = iterate_loop(a0: 0.5, b: 0, delta: 0.5, nSteps: 20);
var distance_to_fp_1 = h1_20.Select(a => Math.Abs(a - 1.0)).ToList();
Console.WriteLine($"Cas 1 : distance au point fixe a*=1.0 sur 20 etapes = {Traj(distance_to_fp_1)}");
Console.WriteLine("  -> convergence monotone vers 0. Point fixe ATTEINT en 1 etape (a0=0.5, a1=1.0).");

// Cas 2 (delta=-0.3) : point fixe theorique = clip(...). Pas de point fixe strict car delta != 0.
// Le clip fait converger vers -1 en quelques etapes.
var h2_20 = iterate_loop(a0: 0.5, b: 0, delta: -0.3, nSteps: 20);
var distance_to_fp_2 = h2_20.Select(a => Math.Abs(a - (-1.0))).ToList();
Console.WriteLine($"Cas 2 : distance au bord -1.0 sur 20 etapes = {Traj(distance_to_fp_2)}");
Console.WriteLine("  -> converge vers 0 (clip impose -1.0 comme limite). Point fixe par clip.");


Cas 1 : distance au point fixe a*=1.0 sur 20 etapes = [0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


  -> convergence monotone vers 0. Point fixe ATTEINT en 1 etape (a0=0.5, a1=1.0).


Cas 2 : distance au bord -1.0 sur 20 etapes = [1.5, 1.2, 0.9, 0.6, 0.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


  -> converge vers 0 (clip impose -1.0 comme limite). Point fixe par clip.


### Lecture de la boucle

**Mesure** :
- **Cas 1** (`delta=0.5`, `a0=0.5`) : point fixe `a* = 1.0` atteint en 1 étape. Convergence triviale — l'addition `0.5 + 0.5` tombe pile sur le point fixe `1.0`, la distance au point fixe passe directement de `0.5` à `0` sans transition.
- **Cas 2** (`delta=-0.3`, `a0=0.5`) : pas de point fixe strict (`delta != 0` -> équation `a* = clip(a* + delta)` n'a pas de solution dans l'intérieur), convergence vers le bord `-1.0` (imposé par le clip). Les distances successives `1.5, 1.2, 0.9, 0.6, 0.3, 0.0` forment une suite arithmétique de pas exactement `0.3 = |delta|` : la **vitesse de convergence EST le paramètre**, et le bord tombe à la 5e étape (0.5 - 5 × 0.3 = -1.0), le clip gelant ensuite la distance à zéro. Le nombre d'étapes d'une trajectoire clippée se prédit avant de lancer la boucle.
- **Cas 3** (`a' = -a`) : oscillation `0.5 -> -0.5 -> 0.5 -> ...`, point fixe trivial `a* = 0` instable.

**Pédagogie** : la phrase ICT << une représentation locale peut modifier le global >> est réalisable, mais **le geste n'aboutit pas toujours à un point fixe**. Le système converge, diverge ou oscille selon la nature de `decision_function`. C'est précisément cette variété qui rend la représentation locale intéressante : on peut choisir la dynamique du global en composant des décisions locales simples.


### Exercice 3 — Changer la lentille, pas la décision

**Contexte.** Les exemples précédents font varier la décision appliquée à la première composante du monde, tandis que la seconde reste spectatrice. Une autre lentille peut exposer une autre partie du même état global.

**Objectif.** Définissez une lentille `L2` sur la seconde composante de `W = (a, b)`, vérifiez ses lois sur plusieurs mondes, puis appliquez la même décision avec `delta = 0.5` à `b`. Vérifiez que la dynamique se déplace vers `b` alors que `a` reste inchangé.

**Indices.**

1. `view_2(w)` renvoie `w[1]`.
2. `update_2(w, p)` conserve `w[0]` et remplace seulement `w[1]`.
3. À chaque étape, composez `view`, `decision_function` et `update` plutôt que de modifier directement le tuple.


In [10]:
// Exercice 3 a completer
// Etape 1 : definir une lentille L2 sur la seconde composante du monde.
// Etape 2 : verifier get-put et put-get sur plusieurs mondes.
// Etape 3 : appliquer la meme decision clippee a b et verifier que a reste inchange.
// Indice : composer L2.view, decision_function et L2.update a chaque etape.

// TODO etudiant : view_2 renvoie world.B ; update_2 conserve world.A et remplace seulement world.B.
static Lens<World, int>? ConstruireL2()
{
    return null;
}

var L2 = ConstruireL2();
var resultat_ex3 = L2;

if (L2 is null)
{
    Console.WriteLine("Exercice 3 a completer");
}
else
{
    var lens = L2;
    foreach (var w in new[] { new World(1, 2), new World(5, 0), new World(-3, 7), new World(0, 0) })
    {
        Console.WriteLine($"w={w} : get-put={get_put_holds(lens, w)}, put-get={put_get_holds(lens, w, 7)}");
    }

    // La meme decision clippee est appliquee a b ; a doit rester inchange.
    int a = 3;
    double b = 0.5;
    for (int k = 0; k < 10; k++)
    {
        var world = new World(a, (int)b);
        int part = (int)decision_function(b, 0.5);
        var monde = lens.Update(world, part);
        a = monde.A;
        b = monde.B;
    }
    Console.WriteLine($"boucle sur b (delta=0.5) : a={a} (inchange), b={b}");
}


Exercice 3 a completer



(8,24): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## Conclusion -- Ce que la dette de vocabulaire solde

**Quatre resultats chiffres sur les open games et lentilles** :

| Section | Mesure | Verdict |
|---|---|---|
| 1 (Lentille) | `L1` (premiere composante) : get-put OK (5 worlds) + put-get OK (25 tests) | lentille CONFORME |
| 1 (Contre-exemple) | `L_bad` : get-put viole (sur 1 cas temoin w=(5,2), suffisant pour falsifier la loi) | la paire n'est PAS une lentille |
| 2 (Composition) | BR chainee `(br_1(F1(x)), br_2(F_compose(x)))` evaluee au cas temoin `x = 0` ; avant composee verifiee sur trois entrees | propriete de Hedges **constatee** sur le cas jouet (pas re-prouvee) |
| 3 (Boucle) | Cas 1 : point fixe `a*=1.0` en 1 etape ; Cas 2 : converge au bord ; Cas 3 : oscillation | variete des dynamiques |

**Ce que ce notebook autorise a dire dans ICT** :

- Une **lentille** n'est pas un mot : c'est une paire `(view, update)` qui satisfait deux lois explicites (`get-put`, `put-get`).
  Sans les lois, on n'a pas une lentille, on a une fonction.
- Un **open game** est composable : la composition preserve la structure de meilleure reponse (propriete de Hedges).
  Composer deux jeux ouverts n'est pas creer un nouveau jeu, c'est deriver la BR du jeu compose par chaining.
- La **boucle** `W -> P -> decision -> update -> W'` peut converger, diverger, ou osciller. Le vocabulaire
  << la representation locale modifie le global >> est realisable ; le **comment** depend du choix de `decision`.

**Ce que ce notebook n'autorise PAS a dire** :

- On n'a PAS re-prouve la theorie des open games (Ghani, Hedges, Winschel & Zahn, 2018). Les resultats sont
  RAPPORTES, verifies numeriquement sur les cas jouets.
- Le modele `OpenGame` ici est un jouet (strategie = 0/1, entree/sortie = floats). Pas une representation
  des vrais open games avec stratigies mixtes, equilibres bayesiens, ou types incertains.
- La boucle `W -> P -> W'` ne montre pas qu'ICT (Integrated Cognitive Theory) peut etre formalisee ici.
  Le vocabulaire est honnete, l'infusion dans ICT reste a faire dans un autre grain.

**Suite logique (hors scope ce cycle)** :

- Un notebook `Lean-21b` ou on formalise la preservation des BR sous composition (item de suite,
  hors scope de ce cycle).
- Une infusion ICT : montrer comment le patron `view -> update -> nouvelle BR` capture un phenomene
  cognitif reel (memoire de travail, integration contexte).
- Une bibliotheque C# `open_games_toy/` qui implemente le modele ci-dessus avec une API plus propre.

**Limites honnement documentees** :

- Implementation `OpenGame` simplifiee (strategies discretes 0/1, pas de types).
- Contre-exemple `L_bad` exhibe sur 1 cas (`w = (5, 2)`) -- suffisant pour falsifier la loi universelle,
  mais on pourrait exhiber la violation sur tous les worlds (trivialement : `update(w, view(w)) = (view(w)+1, w[1]) != w`).
- Pas de profil de performance (open games composition sur 100+ composants).


In [11]:
// Verification rapide : coherence interne du notebook.
// Aucune exception levee : chaque mesure est.imprimee AVANT sa comparaison,
// et le verdict est CONFORME / ECART (calcule=X attendu=Y).
using System.Globalization;
using System.Linq;

static string Verdict(double calcule, double attendu) =>
    Math.Abs(calcule - attendu) < 1e-9
        ? "CONFORME"
        : $"ECART : calcule={calcule.ToString("0.0", CultureInfo.InvariantCulture)} attendu={attendu.ToString("0.0", CultureInfo.InvariantCulture)}";

Console.WriteLine("GT-18 : Open Games et Lentilles (Ghani, Hedges, Winschel & Zahn, 2018)");
Console.WriteLine();
Console.WriteLine("Coherence interne :");

// --- Section 1 : les lois de L1 ---
var worldsCoherence = new[] { new World(1, 2), new World(5, 0), new World(-3, 7), new World(10, -1), new World(0, 0) };
var partsCoherence = new[] { 10, 0, -5, 99, 1 };
bool ok_gp = true, ok_pg = true;
int nbTests = 0;
foreach (var w in worldsCoherence)
{
    if (!get_put_holds(L1, w))
        ok_gp = false;

    foreach (var p in partsCoherence)
    {
        if (!put_get_holds(L1, w, p))
            ok_pg = false;
        nbTests++;
    }
}
Console.WriteLine($"  - L1 (premiere composante) : get-put={ok_gp}, put-get={ok_pg} sur {nbTests} tests");

// --- Section 1 : le contre-exemple L_bad viole get-put sur le cas temoin w = (5, 2) ---
var w_temoin = new World(5, 2);
var w_apres = L_bad.Update(w_temoin, L_bad.View(w_temoin));
bool get_put_violated = !w_apres.Equals(w_temoin);
Console.WriteLine($"  - L_bad (update +1) : w={w_temoin} -> update(w, view(w))={w_apres} ; get-put viole={get_put_violated}");

// --- Section 2 : composition G1 o G2 en serie ---
double x_coh = 0.0;
double y1_coh = G1.Forward(x_coh);
double z_coh = G_compose.Forward(x_coh);
var br_coh = (G1.BestResponse(y1_coh), G2.BestResponse(z_coh));
Console.WriteLine($"  - G1, G2 open games composables en serie : x={S(x_coh)} -> F1={S(y1_coh)} -> F_compose={S(z_coh)} ; br chain=({br_coh.Item1}, {br_coh.Item2})");
Console.WriteLine($"    F_compose(x) : {Verdict(z_coh, 2.0)} (attendu 2.0) ; br chain : {Verdict(br_coh.Item1, 0.0)} / {Verdict(br_coh.Item2, 1.0)}");

// --- Section 3 : les trois dynamiques de reference ---
// referenceCase1 : Cas 1 (delta=0.5, a0=0.5) -> point fixe a*=1.0 atteint en 1 etape.
var h1_coh = iterate_loop(a0: 0.5, b: 0, delta: 0.5, nSteps: 20);
double referenceCase1 = h1_coh[1];
Console.WriteLine($"  - referenceCase1 (Cas 1, delta=0.5, a0=0.5) : a apres 1 etape = {S(referenceCase1)} (attendu 1.0) ; {Verdict(referenceCase1, 1.0)}");
Console.WriteLine($"    distance au point fixe a*=1.0 : {S(Math.Abs(referenceCase1 - 1.0))} (attendu 0.0) ; {Verdict(Math.Abs(referenceCase1 - 1.0), 0.0)}");

// referenceCase2 : Cas 2 (delta=-0.3, a0=0.5) -> convergence vers le bord -1.0, atteint a l'etape 5.
var h2_coh = iterate_loop(a0: 0.5, b: 0, delta: -0.3, nSteps: 20);
int referenceCase2 = -1;
for (int k = 1; k < h2_coh.Count; k++)
{
    if (Math.Abs(h2_coh[k] - (-1.0)) < 1e-9)
    {
        referenceCase2 = k;
        break;
    }
}
Console.WriteLine($"  - referenceCase2 (Cas 2, delta=-0.3, a0=0.5) : bord -1.0 atteint a l'etape {referenceCase2} (attendu 5) ; {Verdict(referenceCase2, 5)}");
Console.WriteLine($"    valeur finale = {S(h2_coh[h2_coh.Count - 1])} (attendu -1.0) ; {Verdict(h2_coh[h2_coh.Count - 1], -1.0)}");

// referenceOscillation : Cas 3 (decision oscillante a' = -a) -> 0.5, -0.5, 0.5, -0.5, 0.5, -0.5, 0.5 sur 6 etapes.
double a_osc = 0.5;
var referenceOscillation = new List<double> { a_osc };
for (int k = 0; k < 6; k++)
{
    a_osc = decision_oscillating(a_osc);
    referenceOscillation.Add(a_osc);
}
Console.WriteLine($"  - referenceOscillation (Cas 3, a' = -a) : {Traj(referenceOscillation)} (attendu [0.5, -0.5, 0.5, -0.5, 0.5, -0.5, 0.5])");
bool oscille = referenceOscillation.Count == 7
    && referenceOscillation.Where((v, i) => i % 2 == 0).All(v => Math.Abs(v - 0.5) < 1e-9)
    && referenceOscillation.Where((v, i) => i % 2 == 1).All(v => Math.Abs(v + 0.5) < 1e-9);
Console.WriteLine($"    oscillation (point fixe trivial a*=0 instable) : {(oscille ? "CONFORME" : "ECART : la suite ne alterne pas entre 0.5 et -0.5")}");
Console.WriteLine();
Console.WriteLine("  - Boucle 3 cas : point fixe / clip / oscillation");


GT-18 : Open Games et Lentilles (Ghani, Hedges, Winschel & Zahn, 2018)


Coherence interne :


  - L1 (premiere composante) : get-put=True, put-get=True sur 25 tests


  - L_bad (update +1) : w=(5, 2) -> update(w, view(w))=(6, 2) ; get-put viole=True


  - G1, G2 open games composables en serie : x=0.0 -> F1=1.0 -> F_compose=2.0 ; br chain=(0, 1)


    F_compose(x) : CONFORME (attendu 2.0) ; br chain : CONFORME / CONFORME


... (3 outputs left out)

    valeur finale = -1.0 (attendu -1.0) ; CONFORME


  - referenceOscillation (Cas 3, a' = -a) : [0.5, -0.5, 0.5, -0.5, 0.5, -0.5, 0.5] (attendu [0.5, -0.5, 0.5, -0.5, 0.5, -0.5, 0.5])


    oscillation (point fixe trivial a*=0 instable) : CONFORME


  - Boucle 3 cas : point fixe / clip / oscillation


**Lecture du controle de coherence -- le notebook se ferme sur son auto-verdict.** La derniere cellule n'enseigne pas un nouveau concept : elle rejoue les quatre acquis de la seance (lois de `L1`, violation de `L_bad` sur le cas temoin `w = (5, 2)`, composition `G1`/`G2` en serie, les trois dynamiques de la boucle) et les confronte a leur verdict.
Chaque ligne resume une mesure deja etablie plus haut ; leur reunion dans une meme sortie joue le role de regression interne : si une modification future du code cassait une loi, c'est ici qu'elle deviendrait visible avant d'atteindre le lecteur.


## References

**Hedges, J. (2017)** -- *Coherence for lenses and open games*. arXiv:1704.02230 (verifie sur arXiv.org : Jules Hedges, 2017-04-07). -> §1 (Lentille, cellules 2-8) : la semantique des lentilles coherentes et le pont vers les open games, sur lequel la Section 1 exhibe la lentille conforme `L1` (25 tests put-get OK) et le contre-exemple `L_bad` (get-put viole sur `w=(5,2)`).

**Ghani, N., Hedges, J., Winschel, V. & Zahn, P. (2018)** -- *Compositional game theory*. LICS 2018, arXiv:1603.04641. **Ancrage fondateur** du carnet. -> §2 (Open games, cellules 9-16) : formalise les open games comme morphismes dans une categorie symetrique monoidale ; la propriete de preservation de la meilleure reponse par composition est le resultat central que la cellule Section 2 verifie numeriquement sur le jouet `G1`/`G2` (BR chainee = composition des BR).

**Bolt, J., Hedges, J. & Zahn, P. (2023)** -- *Bayesian open games*. *Compositionality* 5, DOI 10.32408/compositionality-5-9 (depot arXiv:1910.03656, 2019-10-08). -> §2 (Open games, extension bayesienne documentee) : generalise Ghani et al. 2018 aux environnements stochastiques, choix stochastiques des joueurs et information incomplete. Hors scope du notebook jouet (strategies 0/1, pas de types), mais documente la trajectoire du formalisme au-dela du carnet et fournit le vocabulaire `type` qui distingue l'open game jouet de son extension.

**Riley, M. (2018)** -- *Categories of Optics*. arXiv:1809.00738 (verifie sur arXiv.org : Mitchell Riley, 2018-09-03). -> §1+§2 (fondements categoriques) : formalise les *optics* (generalisation categorique des lentilles) dont Hedges et al. derivent les open games. Pont entre la theorie des categories et l'economique composee ; le notebook s'appuie sur le vocabulaire sans pretendre le formaliser.

**Hedges, J. (2016)** -- *Towards compositional game theory*. These de doctorat, Queen Mary University of London (handle qmro.qmul.ac.uk/123456789/23259). -> §2 (contexte doctoral, fondements longs) : developpe les premieres formulations de la theorie compositionnelle des jeux, dont les articles ulterieurs (Ghani et al. 2018 LICS, Bolt et al. 2023 Compositionality) sont la version deposee et stabilisee.

**Borcherds, R. (2026)** -- *Monster Group, String Theory, Moonshine*, entretien avec C. Jaimungal, Theories of Everything (YouTube). -> §1+§2 (les fondements categoriques, temoignage d'un medaille Fields) : demande ce qu'il trouve « confounding » dans les infinity categories, il repond « the definition, to start off with... I must have read it half a dozen times and half an hour later I just totally forgot what the definition is. Half a dozen times is nowhere near enough » (28:03) -- celui qui a bati des structures algebriques inedites dit son effort devant une definition categorique ; le lecteur qui relit les definitions des Sections 1 et 2 est en bonne compagnie. <https://www.youtube.com/watch?v=xu15ZbxxnUQ&t=1683s>


## À retenir

- **Une lentille n'est pas un mot : c'est une paire `(view, update)`** qui satisfait deux lois explicites — `get-put` et `put-get`. Mesurées ici : `L1` (première composante) est CONFORME (get-put et put-get OK sur 25 tests), tandis que `L_bad` (qui ajoute une constante au lieu de remplacer) viole `get-put` sur le cas témoin `w = (5, 2)`. Sans les lois, on a une fonction, pas une lentille.
- **Un open game est composable** : la composition préserve la structure de meilleure réponse (propriété de Hedges). Composer deux jeux ouverts dérive la BR du jeu composé par chaining — c'est la mesure du §2 sur le jouet `G1`/`G2`.
- **La boucle `W → P → decision → update → W'` a trois dynamiques** : convergence au point fixe (Cas 1, atteint en 1 étape), convergence au bord imposée par le clip (Cas 2), oscillation (Cas 3). Le « comment » dépend du choix de `decision_function`.

## Vérifiez votre compréhension

1. Pourquoi `L_bad` n'est-elle pas une lentille alors que `L1` l'est ? Quelle loi exactement est violée, et sur quel cas témoin ?
2. Dans le Cas 2 (`delta = -0.3`), pourquoi le nombre d'étapes avant d'atteindre le bord `-1.0` se prédit-il analytiquement avant de lancer la boucle ?
3. En quoi la composition de deux open games « dérive » la meilleure réponse du jeu composé plutôt que de la redéfinir ?

<details>
<summary>Éléments de réponse</summary>

1. `L_bad` associe `view_bad(w) = w[0]` à `update_bad(w, p) = (p + 1, w[1])`, qui ajoute 1 au lieu de réinjecter la partie. La loi violée est `get-put` : sur le cas témoin `w = (5, 2)`, `update(w, view(w))` rend `(6, 2)` et non `(5, 2)` (sortie de la cellule 5, reprise par la cellule 25 : « get-put viole sur 1 cas temoin w=(5,2) »). `L1` réinjecte au contraire par `update_first(w, p) = (p, w[1])`, d'où `get-put=True, put-get=True` sur les cas testés (cellule 3), soit `put-get=OK` sur 25 tests (cellule 25). Une seule violation suffit : la loi est universelle, sa violation sur un cas la falsifie (cellule 6).

2. Tant que le clip ne mord pas, l'itération est affine : `a_{n+1} = a_n + delta`, donc `a_n = a_0 + n * delta = 0.5 - 0.3 n`. Le bord `-1.0` est atteint dès que `0.5 - 0.3 n <= -1.0`, soit `n >= 1.5 / 0.3 = 5`. La mesure confirme exactement ce compte : les distances au bord sont `1.5, 1.2, 0.9, 0.6, 0.3, 0.0`, suite arithmétique de pas `0.3 = |delta|`, et le bord tombe à la 5e étape (`0.5 - 5 × 0.3 = -1.0`), le clip gelant ensuite la distance à zéro (sortie de la cellule 20, lecture de la cellule 21).

3. Le code de composition n'écrit aucune nouvelle règle de meilleure réponse : `compose_series` construit `F_compose(x) = g2.forward(g1.forward(x))` et `br_compose` rend la paire `(g1.best_response, g2.best_response)`, c'est-à-dire les BR existantes chaînées. Au cas témoin `x = 0`, la mesure donne `F1(x) = 1.0`, `F_compose(x) = 2.0` et `br chain (s1, s2) = (0, 1)` : `br_1(1.0) = 0` (seuil `y > 0.5`) et `br_2(2.0) = 1` (seuil `z > 1.5`), chaque composant appliquant son propre seuil à la sortie qu'il reçoit réellement (cellule 13). La composition achemine la sortie intermédiaire, elle ne définit pas de nouveau seuil — d'où la formule de la cellule 14 : la BR du composé « n'est pas une nouvelle fonction : c'est la composition point par point des BR des composants ».

</details>

## Pour aller plus loin

- **Hedges (2017)** — *Coherence for lenses and open games* (arXiv:1704.02230) : la sémantique catégorique des lentilles cohérentes et le pont vers les open games.
- **Ghani, Hedges, Winschel & Zahn (2018)** — *Compositional Game Theory* (LICS) : le formalisme fondateur des open games comme morphismes dans une catégorie symétrique monoidale.
- **Bolt, Hedges & Zahn (2023)** — *Bayesian open games* (*Compositionality* 5) : l'extension aux environnements stochastiques et à l'information incomplète.

> Carnet issu de l'EPIC #26534.
